# Question 3: Predicting Next-Month Visa Bulletin Movement

**Research Question:**  
Can historical movement patterns and documented external processing or policy events help predict whether next month's cutoff date will Advance, No Movement, or Retrogress?

This notebook first develops a Historical-Only prediction model using information available from previous Visa Bulletin observations. External event variables will be added in a later step for comparison.

In [1]:
import pandas as pd

q3_df = pd.read_csv(
    "../data/processed/visa_bulletins_transformed_full.csv",
    parse_dates=["bulletin_date", "cutoff_date"]
)

print(q3_df.shape)
q3_df.head()

(6000, 12)


,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_status,cutoff_date,days_behind_bulletin,movement_days,movement_status,months_stagnant,current_streak
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2130,NaN,NaN,0,0
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2160,0.0,No Movement,1,0
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2191,0.0,No Movement,2,0
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2222,0.0,No Movement,3,0
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2250,0.0,No Movement,4,0


## Prediction Target

The goal of Question 3 is to predict the movement status of the next month's Visa Bulletin.

For each visa category, chargeability area, and table type, the next month's movement status is created by shifting the existing `movement_status` variable one month forward.

In [2]:
# Sort observations in chronological order within each Visa Bulletin series

q3_df = q3_df.sort_values(
    by=[
        "visa_category",
        "chargeability_area",
        "table_type",
        "bulletin_date"
    ]
).copy()


# Create the next-month movement target

q3_df["next_month_movement"] = (
    q3_df
    .groupby([
        "visa_category",
        "chargeability_area",
        "table_type"
    ])["movement_status"]
    .shift(-1)
)

In [3]:
print(q3_df["next_month_movement"].value_counts(dropna=False))

next_month_movement
No Movement    3068
Advance        2852
NaN              50
Retrogress       30
Name: count, dtype: int64


In [6]:
# Variables used in the Historical-Only model

historical_features = [
    "visa_category",
    "chargeability_area",
    "table_type",
    "cutoff_status",
    "days_behind_bulletin",
    "movement_days",
    "movement_status",
    "months_stagnant",
    "current_streak"
]

target = "next_month_movement"

# Keep only the variables needed for modeling

historical_model_df = q3_df[
    historical_features + [target, "bulletin_date"]
].copy()

# Remove rows where the target or historical predictors are missing

historical_model_df = historical_model_df.dropna(
    subset=historical_features + [target]
)

print(historical_model_df.shape)
print(historical_model_df[target].value_counts())

(5900, 11)
next_month_movement
No Movement    3043
Advance        2827
Retrogress       30
Name: count, dtype: int64


## Chronological Train-Test Split

Because the goal is to predict future Visa Bulletin movement, the dataset is divided chronologically rather than randomly.

Earlier months are used for training, while the most recent months are kept as unseen test data. This better represents how the model would be used in practice.

In [7]:
# Get all unique bulletin months in chronological order

unique_dates = sorted(historical_model_df["bulletin_date"].unique())

# Use approximately 80% of the months for training

split_index = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_index]

print("Split date:", split_date)

Split date: 2024-09-30 00:00:00


In [ ]:
# Split the dataset using the chronological cutoff

# Create the training dataset using all rows before the split date
train_df = historical_model_df[
    historical_model_df["bulletin_date"] < split_date
].copy()

# Create the testing dataset using the split date and all later rows
test_df = historical_model_df[
    historical_model_df["bulletin_date"] >= split_date
].copy()

# Show the number of rows and columns in the training dataset
print("Training shape:", train_df.shape)

# Show the number of rows and columns in the testing dataset
print("Testing shape:", test_df.shape)

# Show the earliest and latest bulletin dates in the training dataset
print("Training period:",
      train_df["bulletin_date"].min(),
      "to",
      train_df["bulletin_date"].max())


# Show the earliest and latest bulletin dates in the testing dataset
print("Testing period:",
      test_df["bulletin_date"].min(),
      "to",
      test_df["bulletin_date"].max())

Training shape: (4700, 11)
Testing shape: (1200, 11)
Training period: 2016-11-30 00:00:00 to 2024-08-31 00:00:00
Testing period: 2024-09-30 00:00:00 to 2026-08-31 00:00:00


In [18]:
# Select the historical predictor variables from the training dataset
X_train = train_df[historical_features]

# Select the next-month movement target from the training dataset
y_train = train_df[target]

# Select the historical predictor variables from the testing dataset
X_test = test_df[historical_features]

# Select the next-month movement target from the testing dataset
y_test = test_df[target]

# Show the dimensions of the training predictor data
print("X_train:", X_train.shape)

# Show the number of training target values
print("y_train:", y_train.shape)
print("\n")

# Show the dimensions of the testing predictor data
print("X_test:", X_test.shape)

# Show the number of testing target values
print("y_test:", y_test.shape)


# Show how many Advance, No Movement, and Retrogress outcomes are in the test set
print("\n")
print("Testing target distribution:")
print(y_test.value_counts())

print("\n")
print("Training target distribution:")
print(y_train.value_counts())

X_train: (4700, 9)
y_train: (4700,)


X_test: (1200, 9)
y_test: (1200,)


Testing target distribution:
next_month_movement
No Movement    752
Advance        448
Name: count, dtype: int64


Training target distribution:
next_month_movement
Advance        2379
No Movement    2291
Retrogress       30
Name: count, dtype: int64


## Model Preprocessing

The Historical-Only model includes both categorical and numerical predictors.

Categorical variables are converted into numerical indicator variables using one-hot encoding. Numerical variables are standardized so that predictors measured on different scales can be used together in the logistic regression model

In [19]:
# Separate categorical and numerical predictors

categorical_features = [
    "visa_category",
    "chargeability_area",
    "table_type",
    "cutoff_status",
    "movement_status"
]

numeric_features = [
    "days_behind_bulletin",
    "movement_days",
    "months_stagnant",
    "current_streak"
]